In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/rifat4018/
      /kaggle/input/datasets/rifat4018/sat-ext/
        /kaggle/input/datasets/rifat4018/sat-ext/sat_ext/
          README.md
          analysis.py
          audit.py
          config.py
          dataset.py
          dota.py
          ... +10 more files
      /kaggle/input/datasets/rifat4018/sat-splits/
        /kaggle/input/datasets/rifat4018/sat-splits/audit/
        /kaggle/input/datasets/rifat4018/sat-splits/splits/
      /kaggle/input/datasets/rifat4018/sat-splits-extra/
        /kaggle/input/datasets/rifat4018/sat-splits-extra/sat_splits_extra/
      /kaggle/input/datasets/rifat4018/sat-splits-sky/
        /kaggle/input/datasets/rifat4018/sat-splits-sky/audit/
        /kaggle/input/datasets/rifat4018/sat-splits-sky/splits/
      /kaggle/input/datasets/rifat4018/satelite-coco/
        README.dataset.txt
        README.roboflow.txt
        /kaggle/input/datasets/rifat4018/satelite-coco/test/
          1

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/rifat4018/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
utils.warn_if_gpu_wasted()
import roboflow_io as rio
LICENSE = ("Annotations: CC BY 4.0. Imagery: Google Earth, (c) Google and its data providers - research use only, "
           "attribution required; not covered by the CC licence.")
SOURCE = "Screenshots of Google Earth (satellite / aerial imagery), collected and annotated by the authors."
CITATION = "<FILL IN after acceptance>"
for SPLIT in ["native_grouped", "grouped"]:
    try:
        card = rio.datacard(SPLIT, license_text=LICENSE, citation=CITATION, source_text=SOURCE)
    except (FileNotFoundError, AssertionError) as e:
        print("skip", SPLIT, "-", e); continue
    print(card)
    out, z, counts = rio.build_upload_package(SPLIT, readme_text=card)

# Satellite Plane / Ship / Vehicle Detection

Object detection of **planes, ships and vehicles** in satellite imagery, dominated by small and densely packed objects. Companion dataset of the paper *SAHI-Augmented Multi-Architecture Satellite Image Object Detection*.

* Images: **536** (original screenshot size: 1920 x 1080 px (522 images), 2048 x 1152 px (14 images)), annotations: **19738** axis-aligned boxes
* Classes: `plane` (2811), `ship` (1607), `vehicle` (15320)
* Format: COCO JSON (`bbox` = [x, y, width, height] in pixels); YOLO export available from Roboflow
* Split (native_grouped, fingerprint `2c526ef767ce`): train 374 images / 13816 boxes / 364 ground areas, val 80 images / 3105 boxes / 80 ground areas, test 82 images / 2817 boxes / 80 ground areas

## How the split was made
Images that show the same ground area were grouped (Roboflow augmented copies of one source image, byte-identical files, perceptual-hash near-duplicates under flips/rotations, and tiles that overlap acco